In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [3]:
!git clone https://github.com/rahul-ahuja/cd1822-seq-models-transformers-public.git

Cloning into 'cd1822-seq-models-transformers-public'...
remote: Enumerating objects: 171, done.
remote: Counting objects: 100% (171/171), done.
remote: Compressing objects: 100% (140/140), done.
remote: Total 171 (delta 25), reused 167 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (171/171), 1.74 MiB | 9.95 MiB/s, done.
Resolving deltas: 100% (25/25), done.


In [4]:
%cd cd1822-seq-models-transformers-public/project/starter/

/kaggle/working/cd1822-seq-models-transformers-public/project/starter


In [5]:
!pip install -r requirements.txt

INFO: pip is looking at multiple versions of ipykernel to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of ipykernel to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 77.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.4/77.4 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 117.2/117.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 105.1 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 71.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 304.8/304.8 kB 21.3 MB/s eta 0:00:00
  Attempting uninstall: ipykernel
    Found existing installation: ipykernel 6.17.1
    Uninstalling ipykernel-6.17.1:
      Successfully uninstalled ipykernel-6.17.1
ERROR: pip's dependency reso

In [6]:
%cd notebooks/ 

/kaggle/working/cd1822-seq-models-transformers-public/project/starter/notebooks


In [9]:
import os
import sys
import random

import numpy as np
import torch

from datasets import Dataset
from sentence_transformers import (
    SentenceTransformer,
    SentenceTransformerTrainer,
    SentenceTransformerTrainingArguments,
)
from sentence_transformers.losses import MultipleNegativesRankingLoss
from sentence_transformers.training_args import BatchSamplers


In [10]:
# ============================================================
# Configuration
# ============================================================

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

OUTPUT_DIR = "./models/minilm-nq-finetuned"

# Number of NQ queries to load from the repository dataset.
# The original notebook uses 2,000.
NUM_QUERIES = 2000
# Fraction used for fine-tuning.
TRAIN_RATIO = 0.8
NUM_EPOCHS = 3
BATCH_SIZE = 16
LEARNING_RATE = 2e-5
MAX_SEQ_LENGTH = 256
TOP_K = 20

if torch.cuda.is_available():
    device = "cuda"


In [15]:
from beir.datasets.data_loader import GenericDataLoader
from sentence_transformers import SentenceTransformer, InputExample, losses
from torch.utils.data import DataLoader
from beir import util

BASE_MODEL = "all-MiniLM-L6-v2"   # same mini transformer as the starter repo
DATASET_NAME = "nq"
TRAIN_QUERIES = 3000              # number of training (query, positive) pairs to sample
EVAL_QUERIES = 500                # queries used for before/after evaluation (test split)
EPOCHS = 1
BATCH_SIZE = 32
WARMUP_RATIO = 0.1
OUTPUT_DIR = "finetuned-minilm-nq"

In [30]:
# Add src to path for our utilities
sys.path.append('../src')

from data_loader import DataLoader
from evaluator import IRMetrics

# ============================================================
# Load the SAME NQ dataset used by the notebook
# ============================================================

print("1. Loading Natural Questions dataset")

loader = DataLoader("BeIR/nq")

dataset = loader.load_dataset(
    split="test",
    query_sample_size=NUM_QUERIES,
    random_seed=SEED,
)

corpus_texts, query_texts, qrels_dict = (loader.prepare_retrieval_data())

print("\nDataset loaded:")
print(f"  Documents : {len(corpus_texts):,}")
print(f"  Queries   : {len(query_texts):,}")
print(f"  QRELs     : {len(qrels_dict):,}")


# ============================================================
# Create query -> positive document pairs
# ============================================================

print("2. Creating query-document training pairs")

pairs = []
for query_idx, relevant_docs in qrels_dict.items():
    if query_idx >= len(query_texts):
        continue
    query = query_texts[query_idx]

    # NQ may contain more than one relevant document.
    # Create one training example for each relevant document.
    for doc_idx, relevance_score in relevant_docs.items():
        if doc_idx >= len(corpus_texts):
            continue

        document = corpus_texts[doc_idx]

        # Only use genuinely relevant documents.
        if relevance_score > 0:
            pairs.append({"anchor": query, "positive": document,})

print(f"Total positive query-document pairs: {len(pairs):,}")


# ============================================================
# Remove duplicate query-document pairs
# ============================================================

unique_pairs = []
seen = set()

for pair in pairs:
    key = (pair["anchor"], pair["positive"],)

    if key not in seen:
        seen.add(key)
        unique_pairs.append(pair)
        
pairs = unique_pairs

print(f"Unique positive pairs: {len(pairs):,}")

INFO:data_loader:Loading BeIR/nq (Natural Questions) dataset using official BEIR package...
INFO:data_loader:Dataset nq already exists at /kaggle/working/cd1822-seq-models-transformers-public/project/starter/dataset/nq, skipping download...
INFO:data_loader:Loading test split...
INFO:beir.datasets.data_loader:Loading Corpus...


1. Loading Natural Questions dataset


  0%|          | 0/2681468 [00:00<?, ?it/s]

INFO:beir.datasets.data_loader:Loaded 2681468 TEST Documents.
INFO:beir.datasets.data_loader:Doc Example: {'text': "In accounting, minority interest (or non-controlling interest) is the portion of a subsidiary corporation's stock that is not owned by the parent corporation. The magnitude of the minority interest in the subsidiary company is generally less than 50% of outstanding shares, or the corporation would generally cease to be a subsidiary of the parent.[1]", 'title': 'Minority interest'}
INFO:beir.datasets.data_loader:Loading Queries...
INFO:beir.datasets.data_loader:Loaded 3452 TEST Queries.
INFO:beir.datasets.data_loader:Query Example: what is non controlling interest on balance sheet
INFO:data_loader:Sampling 2000 queries and their related documents...
INFO:data_loader:Smart sampling completed:
INFO:data_loader:  - Sampled 2,000 queries from 3,452
INFO:data_loader:  - Included 2,440 related documents from 2,681,468
INFO:data_loader:  - Maintained all 2,000 query-document rela


Dataset loaded:
  Documents : 2,440
  Queries   : 2,000
  QRELs     : 2,000
2. Creating query-document training pairs
Total positive query-document pairs: 2,440
Unique positive pairs: 2,440


In [28]:
train_dataset = Dataset.from_dict(
    {"anchor": [pair["anchor"] for pair in pairs], 
     "positive": [pair["positive"] for pair in pairs],})



print("\n" + "=" * 70)
print("3. Loading MiniLM")
print("=" * 70)

model = SentenceTransformer(MODEL_NAME, device=device,)

model.max_seq_length = MAX_SEQ_LENGTH

print(
    f"Embedding dimension: "
    f"{model.get_sentence_embedding_dimension()}"
)

print(
    f"Maximum sequence length: "
    f"{model.max_seq_length}"
)


# ============================================================
# Define contrastive retrieval loss
# ============================================================

print("\n" + "=" * 70)
print("4. Configuring contrastive retrieval training")
print("=" * 70)

loss = MultipleNegativesRankingLoss(model)

print("Loss: MultipleNegativesRankingLoss")

print(
    "\nEach query is paired with a relevant document."
)

print(
    "Other documents in the batch act as negatives."
)


# ============================================================
# Training arguments
# ============================================================

use_fp16 = torch.cuda.is_available()

training_args = SentenceTransformerTrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    fp16=use_fp16,
    bf16=False,
    # Important for contrastive retrieval training.
    batch_sampler=BatchSamplers.NO_DUPLICATES,
    logging_steps=10,
    save_strategy="epoch",
    save_total_limit=2,
    seed=SEED,
    report_to="none",)


# ============================================================
# Trainer
# ============================================================

trainer = SentenceTransformerTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    loss=loss,)


# ============================================================
# Fine-tune
# ============================================================

print("5. Fine-tuning MiniLM")

trainer.train()

print("6. Saving fine-tuned model")

os.makedirs(OUTPUT_DIR, exist_ok=True,)

model.save_pretrained(OUTPUT_DIR)

INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/modules.json "HTTP/1.1 200 OK"



3. Loading MiniLM


INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence-transformers/all-MiniLM-L6-v2/1110a243fdf4706b3f48f1d95db1a4f5529b4d41/config_sentence_transformers.json "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/README.md "H

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
INFO:httpx:HTTP Request: HEAD https://huggingface.co/sentenc

...

INFO:sentence_transformers.base.trainer:Saving model checkpoint to finetuned-minilm-nq/checkpoint-117
INFO:sentence_transformers.base.model:Saving model to finetuned-minilm-nq/checkpoint-117


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INFO:sentence_transformers.base.model:Saving model to finetuned-minilm-nq


6. Saving fine-tuned model


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [29]:
from transformer_retriever import TransformerRetriever
from evaluator import IRMetrics

transformer_retriever = TransformerRetriever(model_name=OUTPUT_DIR)
transformer_retriever.build_index(corpus_texts)

# Run semantic retrieval
transformer_results = transformer_retriever.retrieve(query_texts, k=20)

# Evaluate performance
transformer_metrics = IRMetrics.evaluate_retrieval(transformer_results, qrels_dict)
IRMetrics.print_metrics(transformer_metrics, "Transformer (Semantic) Results")

INFO:sentence_transformers.base.model:No device provided, using cuda:0
INFO:sentence_transformers.base.model:Loading SentenceTransformer model from finetuned-minilm-nq.


🤖 Loading transformer model: finetuned-minilm-nq


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/finetuned-minilm-nq "HTTP/1.1 401 Unauthorized"


✅ Model loaded: finetuned-minilm-nq
🧠 Building semantic index...


Batches:   0%|          | 0/77 [00:00<?, ?it/s]

✅ Semantic index built for 2,440 documents
   Embedding dimension: 384
🔍 Running semantic retrieval for 2000 queries...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

✅ Retrieved top-20 documents using semantic similarity

📊 Transformer (Semantic) Results
Recall@1    : 0.8092
Recall@5    : 0.9730
Recall@10   : 0.9845
Precision@5 : 0.2367
MRR         : 0.9397
